In [1]:
# 01_data_construction.ipynb -- merge public financials + agency ratings into the modeling corpus

# === Shared configuration (self-contained: one code cell per notebook) ===
from __future__ import annotations
import os, sys, json, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Resolve the repository root (the folder that holds both 'results' and 'notebooks').
def _find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "results").is_dir() and (cand / "notebooks").is_dir():
            return cand
    return start.resolve().parent if start.resolve().name == "notebooks" else start.resolve()

ROOT = _find_root(Path.cwd())
DATA, RAW, PROC = ROOT / "data", ROOT / "data" / "raw", ROOT / "data" / "processed"
RESULTS, FIGS, TABS = ROOT / "results", ROOT / "results" / "figures", ROOT / "results" / "tables"
for d in (DATA, RAW, PROC, RESULTS, FIGS, TABS):
    d.mkdir(parents=True, exist_ok=True)

# Load environment variables (.env is git-ignored; never commit secrets).
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except Exception:
    pass
LLM_MODELS = {
    "weak":   os.getenv("LLM_MODEL_WEAK", ""),
    "mid":    os.getenv("LLM_MODEL_MID", ""),
    "strong": os.getenv("LLM_MODEL_STRONG", ""),
}
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

# Plotting convention: seaborn, grayscale, no caption, 600 dpi, saved as PNG and PDF.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(context="paper", style="ticks")
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 600, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.grid": True, "grid.alpha": 0.3, "image.cmap": "Greys",
})

def gray_palette(n: int):
    # Evenly spaced grays (mid -> dark) for categorical series.
    return sns.color_palette("Greys", max(n + 2, 3))[2:n + 2]

def save_fig(fig, name: str):
    # Save a figure as PNG and PDF at 600 dpi; no caption is added here.
    for ext in ("png", "pdf"):
        fig.savefig(FIGS / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

def save_table(df: pd.DataFrame, name: str, index: bool = False):
    # Persist a results table as CSV and (best-effort) LaTeX for the manuscript.
    df.to_csv(TABS / f"{name}.csv", index=index)
    try:
        (TABS / f"{name}.tex").write_text(df.to_latex(index=index))
    except Exception:
        pass

# Fine-grained rating scale (low -> high); used to map labels to an ordinal rank.
RATING_SCALE = ["D","C","CC","CCC-","CCC","CCC+","B-","B","B+","BB-","BB","BB+",
                "BBB-","BBB","BBB+","A-","A","A+","AA-","AA","AA+","AAA"]
RANK = {g: i for i, g in enumerate(RATING_SCALE)}
def to_rank(label):
    # Map a rating label to an ordinal rank; returns np.nan if unmatched.
    if not isinstance(label, str):
        return np.nan
    s = label.strip().upper().replace(" ", "")
    return RANK.get(s, np.nan)

# --- 01: build data/processed/corpus.parquet from public raw sources ---
# Expected raw inputs (see data/README.md for acquisition; not redistributed here):
#   data/raw/corporate_rating.csv   Kaggle: financial ratios + agency ratings (S&P/Moody's/Fitch/Egan-Jones)
#   data/raw/ccrd/                  CreditARF/CCRD: financials + annual-report text (optional, for context)
kaggle_csv = RAW / "corporate_rating.csv"
if not kaggle_csv.exists():
    print("[skip] missing:", kaggle_csv.relative_to(ROOT))
    print("Add the raw datasets per data/README.md, then re-run this notebook.")
else:
    df = pd.read_csv(kaggle_csv)
    df.columns = [c.strip() for c in df.columns]
    rating_col = next((c for c in df.columns if c.lower() == "rating"), df.columns[0])
    df["rating_rank"] = df[rating_col].map(to_rank)
    before = len(df)
    df = df.dropna(subset=["rating_rank"]).reset_index(drop=True)
    df["rating_rank"] = df["rating_rank"].astype(int)
    # Numeric feature columns = everything numeric except the target rank.
    feat = [c for c in df.select_dtypes("number").columns if c != "rating_rank"]
    df.to_parquet(PROC / "corpus.parquet")
    (PROC / "feature_columns.json").write_text(json.dumps(feat, indent=2))
    dist = df[rating_col].value_counts().rename_axis("rating").reset_index(name="n")
    save_table(dist, "01_rating_distribution")
    print(f"kept {len(df)}/{before} rows | {len(feat)} numeric features")
    print("wrote:", (PROC / "corpus.parquet").relative_to(ROOT))


kept 2029/2029 rows | 25 numeric features
wrote: data\processed\corpus.parquet
